# Calmar ratio exploration

Calmar ratio is annualized compounded return divided by the magnitude of maximum drawdown. Contributions are removed before daily returns are compounded, so deposits do not hide losses.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from retail_sp500.daily_data import daily_data_summary, load_or_fetch_twelve_data_daily

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

from retail_sp500.limit_plotting import (
    calmar_by_discount_figure,
    strategy_calmar_ranking_figure,
    strategy_drawdown_figure,
    strategy_return_drawdown_figure,
    strategy_wealth_figure,
)
from retail_sp500.limit_portfolio import (
    RecurringLimitConfig,
    compare_recurring_limit_strategies,
    evaluate_recurring_limit_grid,
)

In [ ]:
SYMBOL = "SPY"
START_DATE = "2007-06-01"
CACHE_PATH = Path("data/processed/spy_daily_1day.csv")
REFRESH = False

daily = load_or_fetch_twelve_data_daily(
    os.getenv("TWELVE_DATA_API_KEY"),
    cache_path=CACHE_PATH,
    refresh=REFRESH,
    symbol=SYMBOL,
    start_date=START_DATE,
)

source = daily_data_summary(daily, symbol=SYMBOL)
print(source)
assert source["interval"] == "1day"
assert daily.index.max() <= pd.Timestamp.today().normalize()
daily.tail()

## Calmar sensitivity across discounts and expiry horizons

In [ ]:
DISCOUNTS = np.arange(0.0, 0.0501, 0.001)
WAIT_HORIZONS = [1, 3, 5, 10, 20]

grid = pd.concat(
    [
        evaluate_recurring_limit_grid(
            daily,
            DISCOUNTS,
            max_wait_sessions=wait,
            initial_cash=100_000,
            monthly_contribution=1_000,
        ).assign(wait_horizon=wait)
        for wait in WAIT_HORIZONS
    ],
    ignore_index=True,
)

grid[[
    "discount",
    "wait_horizon",
    "annualized_return",
    "max_drawdown",
    "calmar_ratio",
    "ending_excess_value",
]].sort_values("calmar_ratio", ascending=False).head(25)

In [ ]:
calmar_by_discount_figure(grid).show()

## Compare top Calmar strategies with simple rules

In [ ]:
top = grid.dropna(subset=["calmar_ratio"]).sort_values("calmar_ratio", ascending=False).head(3)
strategies = {
    "0.5% below, 5 sessions": RecurringLimitConfig(0.005, 5, 100_000, 1_000),
    "1.0% below, 5 sessions": RecurringLimitConfig(0.010, 5, 100_000, 1_000),
}
for rank, row in enumerate(top.itertuples(index=False), start=1):
    strategies[f"Calmar rank {rank}: {row.discount:.1%} below, {row.wait_horizon} sessions"] = (
        RecurringLimitConfig(float(row.discount), int(row.wait_horizon), 100_000, 1_000)
    )

metrics, curves = compare_recurring_limit_strategies(daily, strategies)
metrics.sort_values("calmar_ratio", ascending=False)

In [ ]:
strategy_calmar_ranking_figure(metrics).show()

In [ ]:
strategy_return_drawdown_figure(metrics).show()

In [ ]:
strategy_wealth_figure(curves).show()

In [ ]:
strategy_drawdown_figure(curves).show()

## Limitations

Calmar is dominated by one historical worst drawdown and can change materially with the sample. This implementation excludes dividends, cash yield, spreads, fees, taxes, and SGD/USD effects.